# Clean Geofabrik Food Venues

This notebook contains section 1.3 of the prospecting workflow: inspect OSM tags, enrich blank fields, map restaurant/fast-food cuisine and dish labels using the merged mapping CSV, drop raw JSON tags, and save the cleaned dataset for the prospecting notebook.

Run this notebook before `../france_food_delivery_prospects.ipynb`. The mapping is loaded from `cuisine_mapping_merged_v2.csv`, which already includes the reviewed and resolved decisions for the current Geofabrik extract.

In [314]:
import json
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd()
if DATA_DIR.name != "france_geofabrik_raw_data":
    DATA_DIR = DATA_DIR / "france_geofabrik_raw_data"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Geofabrik data folder not found: {DATA_DIR}")

pbf_df = pd.read_csv(
    DATA_DIR / "food_venues_from_pbf.csv",
    sep=";", low_memory=False, dtype={"osm_id": "string"}
)
print(f"Loaded Geofabrik food venues: {len(pbf_df):,} rows, {len(pbf_df.columns)} columns")
display(pbf_df.columns.to_list())

Loaded Geofabrik food venues: 190,259 rows, 22 columns


['X',
 'Y',
 'osm_id',
 'type',
 'name',
 'brand',
 'operator',
 'wheelchair',
 'opening_hours',
 'level',
 'siret',
 'profession_ref',
 'wikidata',
 'website',
 'phone',
 'email',
 'facebook',
 'address',
 'com_insee',
 'com_nom',
 'last_update',
 'tags']

#### 1.3.1 Inspect OSM tags

Compare common Geofabrik extract fields with their corresponding OSM tag values and review which tags are most common.

In [315]:
tag_records = pbf_df["tags"].map(lambda value: json.loads(value) if pd.notna(value) else {})
tag_key_counts = Counter(
    key for record in tag_records for key, value in record.items() if str(value).strip()
)

tag_column_map = {
    "type": ["amenity", "shop"],
    "name": ["name"],
    "brand": ["brand"],
    "operator": ["operator"],
    "wheelchair": ["wheelchair"],
    "opening_hours": ["opening_hours"],
    "level": ["level"],
    "siret": ["ref:FR:SIRET"],
    "wikidata": ["wikidata"],
    "website": ["website", "contact:website"],
    "phone": ["phone", "contact:phone"],
    "email": ["email", "contact:email"],
    "facebook": ["facebook", "contact:facebook"],
    "com_nom": ["addr:city"],
}
tag_fields = {
    key: pd.Series((record.get(key, "") for record in tag_records), index=pbf_df.index)
        .fillna("").astype(str).str.strip().str.casefold()
    for key in {key for keys in tag_column_map.values() for key in keys}
    if key in tag_key_counts
}
comparison_rows = []
for column, tag_keys in tag_column_map.items():
    column_values = pbf_df[column].fillna("").astype(str).str.strip().str.casefold()
    candidates = [tag_fields[key] for key in tag_keys if key in tag_fields]
    tag_present = pd.Series(False, index=pbf_df.index)
    matches_tag = pd.Series(False, index=pbf_df.index)
    for candidate in candidates:
        tag_present |= candidate.ne("")
        matches_tag |= candidate.eq(column_values) & candidate.ne("")
    both_present = column_values.ne("") & tag_present
    comparison_rows.append({
        "extract column": column,
        "tag key(s)": ", ".join(tag_keys),
        "column values": int(column_values.ne("").sum()),
        "tag values": int(tag_present.sum()),
        "both present": int(both_present.sum()),
        "equal to a tag value": int((both_present & matches_tag).sum()),
        "different from all tag values": int((both_present & ~matches_tag).sum()),
    })




In [316]:
display(pd.DataFrame(comparison_rows).set_index("extract column"))

,tag key(s),column values,tag values,both present,equal to a tag value,different from all tag values
extract column,,,,,,
type,"amenity, shop",190259,190259,190259,190259,0
name,name,179576,179577,179576,179576,0
brand,brand,11766,11766,11766,11766,0
operator,operator,4267,4267,4267,4267,0
wheelchair,wheelchair,29650,29650,29650,29650,0
opening_hours,opening_hours,70401,70401,70401,70401,0
level,level,8143,8143,8143,8143,0
siret,ref:FR:SIRET,0,67299,0,0,0
wikidata,wikidata,517,517,517,517,0


In [317]:
print(f"Distinct OSM tag keys in this extract: {len(tag_key_counts):,}")
display(pd.Series(dict(tag_key_counts.most_common(15)), name="tagged rows").to_frame())

Distinct OSM tag keys in this extract: 1,916


,tagged rows
name,179577
amenity,161068
cuisine,76923
opening_hours,70401
ref:FR:SIRET,67299
addr:street,63359
addr:housenumber,59392
phone,57013
source,47192
addr:postcode,45023


#### 1.3.2 Enrich blank Geofabrik fields from OSM tags

Fill only blank Geofabrik values from the corresponding OSM tags. For addresses, use `addr:full` when available; otherwise assemble available house number, street, postcode, and city. BANCO is not modified.

In [318]:
# Return the first non-empty OSM tag from the preferred-key list.
def first_tag_value(record, keys):
    for key in keys:
        value = str(record.get(key, "")).strip()
        if value:
            return value
    return ""

In [319]:
# Fill missing fields from OSM tags and construct fallback addresses.
tag_fill_counts = {}

tag_field_map = {
    "name": ["name"],
    "brand": ["brand"],
    "operator": ["operator"],
    "phone": ["phone", "contact:phone"],
    "website": ["website", "contact:website"],
    "email": ["email", "contact:email"],
    "facebook": ["facebook", "contact:facebook"],
    "siret": ["ref:FR:SIRET"],
    "com_nom": ["addr:city"],
}

for column, tag_keys in tag_field_map.items():
    tag_values = tag_records.map(lambda record: first_tag_value(record, tag_keys))
    current_values = pbf_df[column].fillna("").astype(str).str.strip()
    fill_mask = current_values.eq("") & tag_values.ne("")
    pbf_df.loc[fill_mask, column] = tag_values.loc[fill_mask]
    tag_fill_counts[column] = int(fill_mask.sum())

full_address = tag_records.map(lambda record: first_tag_value(record, ["addr:full"]))
address_parts = pd.DataFrame({
    "house_number": tag_records.map(lambda record: first_tag_value(record, ["addr:housenumber"])),
    "street": tag_records.map(lambda record: first_tag_value(record, ["addr:street"])),
    "postcode": tag_records.map(lambda record: first_tag_value(record, ["addr:postcode"])),
    "city": tag_records.map(lambda record: first_tag_value(record, ["addr:city"])),
}, index=pbf_df.index)
street_address = (address_parts["house_number"] + " " + address_parts["street"]).str.strip()
locality = (address_parts["postcode"] + " " + address_parts["city"]).str.strip()
has_street_address = street_address.ne("")
assembled_address = street_address.where(has_street_address, locality)
add_locality = has_street_address & locality.ne("")
assembled_address.loc[add_locality] = street_address.loc[add_locality] + ", " + locality.loc[add_locality]
address_from_tags = full_address.where(full_address.ne(""), assembled_address)
current_address = pbf_df["address"].fillna("").astype(str).str.strip()
address_fill_mask = current_address.eq("") & address_from_tags.ne("")
pbf_df.loc[address_fill_mask, "address"] = address_from_tags.loc[address_fill_mask]
tag_fill_counts["address"] = int(address_fill_mask.sum())


In [320]:
# Summarize how many blank fields were populated from OSM tags.
display(pd.Series(tag_fill_counts, name="Geofabrik blank values filled from tags").to_frame())

,Geofabrik blank values filled from tags
name,1
brand,0
operator,0
phone,12934
website,5668
email,3339
facebook,12470
siret,67299
com_nom,0
address,0


#### 1.3.3 Extract and clean cuisine values

Preserve the original cuisine tag, split only on semicolons, normalize tokens for matching, map each token independently, and keep cuisines and dishes in separate venue fields. Optional audit and regional matching follow below.


##### 1.3.3.1 Extract and inspect raw cuisine values

Keep the original OSM tag unchanged before any cleaning.


In [321]:
# Extract each eligible venue's original OSM cuisine tag unchanged.
cuisine_eligible = pbf_df["type"].astype("string").str.casefold().isin(["restaurant", "fast_food"])
raw_cuisine_values = tag_records.map(lambda record: record.get("cuisine", pd.NA))
raw_cuisine_values = raw_cuisine_values.map(
    lambda value: pd.NA if pd.isna(value) or not str(value).strip() else str(value)
).astype("string")
pbf_df["cuisine_raw"] = raw_cuisine_values.where(cuisine_eligible)
raw_cuisine_venue_count = int(pbf_df["cuisine_raw"].notna().sum())
print(f"Restaurant/fast-food venues with a raw cuisine tag: {raw_cuisine_venue_count:,}")


Restaurant/fast-food venues with a raw cuisine tag: 71,871


In [322]:
# Display extracted raw cuisine tag strings and their venue counts before cleaning.
extracted_cuisine_values = (
    pbf_df.loc[cuisine_eligible, "cuisine_raw"]
    .dropna()
    .value_counts()
    .rename_axis("cuisine_raw")
    .rename("venues")
    .to_frame()
)
print(f"Distinct extracted raw cuisine strings: {len(extracted_cuisine_values):,}")
display(extracted_cuisine_values)

Distinct extracted raw cuisine strings: 5,001


,venues
cuisine_raw,
pizza,10718
french,9485
regional,4777
burger,4550
kebab,3309
...,...
italian;pizza;regional;french,1
pizza;crepe;salad;french,1
vietnamese;french,1


##### 1.3.3.2 Split multivalue cuisine tags

Split only on semicolons; retain underscores and commas inside tokens.


In [323]:
# Split cuisine tags on semicolons only; underscores and commas remain part of each token.
cuisine_tokens = (
    pbf_df.loc[cuisine_eligible, ["cuisine_raw"]]
    .dropna(subset=["cuisine_raw"])
    .assign(cuisine_token=lambda frame: frame["cuisine_raw"].str.split(";"))
    .explode("cuisine_token")
    .assign(cuisine_token=lambda frame: frame["cuisine_token"].str.strip())
    .loc[lambda frame: frame["cuisine_token"].ne("")]
    .rename_axis("venue_id")
    .reset_index()
)
print(f"Extracted cuisine tokens after semicolon splitting: {len(cuisine_tokens):,}")
display(cuisine_tokens[["cuisine_raw", "cuisine_token"]].head(100))


Extracted cuisine tokens after semicolon splitting: 90,489


,cuisine_raw,cuisine_token
0,burger,burger
1,french,french
2,regional;french,regional
3,regional;french,french
4,regional,regional
...,...,...
95,bistro;french,french
96,french,french
97,french,french
98,french,french


##### 1.3.3.3 Normalize tokens for matching

Trim whitespace, normalize Unicode, and casefold lookup keys while preserving source values.


In [324]:
# Build a mapping key for each token while retaining the raw tag and split token.
def normalize_cuisine_value(value):
    return " ".join(unicodedata.normalize("NFC", str(value)).split()).casefold()

cuisine_tokens["cuisine_key"] = cuisine_tokens["cuisine_token"].map(normalize_cuisine_value)
display(cuisine_tokens[["cuisine_raw", "cuisine_token", "cuisine_key"]].head(100))


,cuisine_raw,cuisine_token,cuisine_key
0,burger,burger,burger
1,french,french,french
2,regional;french,regional,regional
3,regional;french,french,french
4,regional,regional,regional
...,...,...,...
95,bistro;french,french,french
96,french,french,french
97,french,french,french
98,french,french,french


##### 1.3.3.4 Map tokens and build venue cuisine and dish fields

Map each token by reviewed status, canonical value, and category; aggregate cuisine and dish separately.


In [325]:
# Map each split token to its reviewed status, canonical value, and output category.
cuisine_mapping = pd.read_csv(
    DATA_DIR / "cuisine_mapping_v2.csv",
    dtype="string",
).fillna("")
required_columns = {"raw_value", "canonical_value", "status", "category_type"}
if not required_columns.issubset(cuisine_mapping.columns):
    raise ValueError(f"Cuisine mapping must contain {sorted(required_columns)}")
cuisine_mapping["cuisine_key"] = cuisine_mapping["raw_value"].map(normalize_cuisine_value)
cuisine_mapping["status"] = cuisine_mapping["status"].str.strip().str.casefold()
cuisine_mapping["category_type"] = cuisine_mapping["category_type"].str.strip().str.casefold()
if cuisine_mapping["cuisine_key"].eq("").any() or cuisine_mapping["cuisine_key"].duplicated().any():
    raise ValueError("Cuisine mapping has blank or duplicate normalized raw_value entries")

mapping_lookup = cuisine_mapping[
    ["cuisine_key", "raw_value", "canonical_value", "status", "category_type"]
].rename(columns={"raw_value": "mapping_raw_value"})
cuisine_token_mapping = cuisine_tokens.merge(
    mapping_lookup,
    on="cuisine_key",
    how="left",
    validate="many_to_one",
)
cuisine_token_mapping["mapping_status"] = cuisine_token_mapping["status"].where(
    cuisine_token_mapping["status"].ne(""), "unmapped"
)

mapped_tokens = cuisine_token_mapping.loc[
    cuisine_token_mapping["status"].eq("mapped")
].copy()
if not mapped_tokens["category_type"].isin(["cuisine", "dish"]).all():
    raise ValueError("Mapped cuisine entries must use category_type 'cuisine' or 'dish'")

def join_unique_canonical(values):
    return ";".join(dict.fromkeys(value for value in values if value))

cuisine_by_venue = (
    mapped_tokens.loc[mapped_tokens["category_type"].eq("cuisine")]
    .groupby("venue_id")["canonical_value"]
    .agg(join_unique_canonical)
)
dish_by_venue = (
    mapped_tokens.loc[mapped_tokens["category_type"].eq("dish")]
    .groupby("venue_id")["canonical_value"]
    .agg(join_unique_canonical)
)
pbf_df["cuisine_standard"] = cuisine_by_venue.reindex(pbf_df.index).astype("string")
pbf_df["dish"] = dish_by_venue.reindex(pbf_df.index).astype("string")

print("Token mapping status counts:")
display(cuisine_token_mapping["mapping_status"].value_counts().to_frame("tokens"))
display(cuisine_token_mapping[[
    "cuisine_raw", "cuisine_token", "cuisine_key", "canonical_value",
    "status", "category_type", "mapping_status"
]].head(100))


Token mapping status counts:


,tokens
mapping_status,
mapped,88423
invalid,2066


,cuisine_raw,cuisine_token,cuisine_key,canonical_value,status,category_type,mapping_status
0,burger,burger,burger,burger,mapped,dish,mapped
1,french,french,french,french,mapped,cuisine,mapped
2,regional;french,regional,regional,regional,mapped,cuisine,mapped
3,regional;french,french,french,french,mapped,cuisine,mapped
4,regional,regional,regional,regional,mapped,cuisine,mapped
...,...,...,...,...,...,...,...
95,bistro;french,french,french,french,mapped,cuisine,mapped
96,french,french,french,french,mapped,cuisine,mapped
97,french,french,french,french,mapped,cuisine,mapped
98,french,french,french,french,mapped,cuisine,mapped


##### 1.3.3.5 Review mapped token results

Inspect approved tokens, their canonical labels, categories, and venue counts.


In [326]:
# Display each approved token with its canonical mapping and category.
mapped_token_summary = (
    cuisine_token_mapping.loc[cuisine_token_mapping["status"].eq("mapped")]
    .groupby(
        ["cuisine_token", "cuisine_key", "canonical_value", "category_type"],
        dropna=False,
    )["venue_id"]
    .nunique()
    .rename("venues")
    .reset_index()
    .sort_values(["category_type", "venues", "cuisine_token"], ascending=[True, False, True])
    .reset_index(drop=True)
)
print(f"Distinct approved token-to-canonical mappings: {len(mapped_token_summary):,}")
display(mapped_token_summary)


Distinct approved token-to-canonical mappings: 821


,cuisine_token,cuisine_key,canonical_value,category_type,venues
0,french,french,french,cuisine,12712
1,regional,regional,regional,cuisine,6687
2,italian,italian,italian,cuisine,4175
3,asian,asian,asian,cuisine,2846
4,japanese,japanese,japanese,cuisine,2523
...,...,...,...,...,...
816,waffer,waffer,waffle,dish,1
817,youfka,youfka,yufka,dish,1
818,渝小面,渝小面,chongqing_noodle,dish,1
819,火鍋,火鍋,hotpot,dish,1


In [327]:
# Review approved split tokens beside their canonical mapped value and output category.
mapped_token_summary = (
    cuisine_token_mapping.loc[cuisine_token_mapping["status"].eq("mapped")]
    .groupby(
        ["cuisine_token", "cuisine_key", "canonical_value", "category_type"],
        dropna=False,
    )["venue_id"]
    .nunique()
    .rename("venues")
    .reset_index()
    .sort_values(["category_type", "venues", "cuisine_token"], ascending=[True, False, True])
    .reset_index(drop=True)
)
print(f"Distinct approved token-to-canonical mappings: {len(mapped_token_summary):,}")
display(mapped_token_summary)


Distinct approved token-to-canonical mappings: 821


,cuisine_token,cuisine_key,canonical_value,category_type,venues
0,french,french,french,cuisine,12712
1,regional,regional,regional,cuisine,6687
2,italian,italian,italian,cuisine,4175
3,asian,asian,asian,cuisine,2846
4,japanese,japanese,japanese,cuisine,2523
...,...,...,...,...,...
816,waffer,waffer,waffle,dish,1
817,youfka,youfka,yufka,dish,1
818,渝小面,渝小面,chongqing_noodle,dish,1
819,火鍋,火鍋,hotpot,dish,1


##### 1.3.3.6 Optional token audit export

This audit is optional and its code is commented out by default. Uncomment the code cell only when you need to recreate the audit CSV.


In [328]:
# Optional audit export. Uncomment these lines to recreate cuisine_mapping_audit.csv.
# # Export every token, including invalid and unmapped entries, for review.
# cuisine_mapping_audit = cuisine_token_mapping[[
#     "venue_id",
#     "cuisine_raw",
#     "cuisine_token",
#     "cuisine_key",
#     "mapping_raw_value",
#     "canonical_value",
#     "category_type",
#     "status",
#     "mapping_status",
# ]].rename(columns={
#     "canonical_value": "mapping_result",
#     "category_type": "category",
# })
# audit_path = DATA_DIR / "cuisine_mapping_audit.csv"
# cuisine_mapping_audit.to_csv(audit_path, index=False, encoding="utf-8-sig")
# review_tokens = cuisine_mapping_audit[
#     cuisine_mapping_audit["mapping_status"].isin(["invalid", "unmapped"])
# ]
# print(f"Exported {len(cuisine_mapping_audit):,} token rows to {audit_path}")
# print("Invalid and unmapped token rows retained for review:")
# display(review_tokens.head(20))


##### 1.3.3.7 Check cuisine and dish fields on venues

Confirm mapped cuisine and dish values are present on the main venue dataframe.


In [329]:
# Verify that mapped cuisine tokens have been aggregated back onto each venue.
venue_cuisine_columns = [
    column for column in ["osm_id", "name", "type", "cuisine_raw", "cuisine_standard", "dish"]
    if column in pbf_df.columns
]
venue_cuisine_preview = pbf_df.loc[
    cuisine_eligible & pbf_df["cuisine_raw"].notna(),
    venue_cuisine_columns,
]
print(f"Venues with raw cuisine values mapped into the main dataframe: {len(venue_cuisine_preview):,}")
display(venue_cuisine_preview)


Venues with raw cuisine values mapped into the main dataframe: 71,871


,osm_id,name,type,cuisine_raw,cuisine_standard,dish
0,node/960852,Burger King,fast_food,burger,<NA>,burger
2,node/25733700,L'Esprit Bistrot,restaurant,french,french,<NA>
3,node/26509939,Au Vieux Moulin,restaurant,regional;french,regional;french,<NA>
5,node/26858709,Maison Las Clauzes,restaurant,regional,regional,<NA>
7,node/29047143,Bistr'O bo bun,restaurant,vietnamese;japanese,vietnamese;japanese,<NA>
...,...,...,...,...,...,...
190251,way/1562674369,Hippopotamus,restaurant,steak_house,<NA>,steak
190253,way/1563095366,Kiosque 10 - Le Bateguier,fast_food,sandwich,<NA>,sandwich
190254,way/1563095367,Kiosque 5 - Le V,fast_food,sandwich,<NA>,sandwich
190255,way/1563095368,Kiosque 3 - Chez Bruno,fast_food,sandwich,<NA>,sandwich


##### 1.3.3.8 Match cuisines to the regional mapping

Attach continent, region, country, normalized cuisine, and locality for each venue-cuisine pair.


In [330]:
# Match each venue-cuisine pair separately so regional metadata stays linked to its cuisine.
region_mapping = pd.read_csv(
    DATA_DIR / "cuisine_region_mapping.csv",
    dtype="string",
).fillna("")
region_columns = {
    "cuisine",
    "normalized_continent",
    "normalized_region",
    "normalized_country",
    "normalized_cuisine",
    "normalized_locality",
}
if not region_columns.issubset(region_mapping.columns):
    raise ValueError(f"cuisine_region_mapping.csv must contain {sorted(region_columns)}")
region_mapping["cuisine_key"] = region_mapping["cuisine"].map(normalize_cuisine_value)
if region_mapping["cuisine_key"].eq("").any() or region_mapping["cuisine_key"].duplicated().any():
    raise ValueError("cuisine_region_mapping.csv has blank or duplicate normalized cuisine keys")

venue_cuisine_pairs = (
    mapped_tokens.loc[
        mapped_tokens["status"].eq("mapped")
        & mapped_tokens["category_type"].eq("cuisine"),
        ["venue_id", "canonical_value"],
    ]
    .rename(columns={"canonical_value": "cuisine"})
    .drop_duplicates(["venue_id", "cuisine"])
    .assign(cuisine_key=lambda frame: frame["cuisine"].map(normalize_cuisine_value))
)
venue_details = pbf_df[[column for column in ["osm_id", "name", "type"] if column in pbf_df.columns]].copy()
venue_details["venue_id"] = pbf_df.index
region_lookup = region_mapping.rename(columns={"cuisine": "region_mapping_cuisine"})
venue_cuisine_region = (
    venue_cuisine_pairs
    .merge(
        region_lookup,
        on="cuisine_key",
        how="left",
        validate="many_to_one",
    )
    .merge(venue_details, on="venue_id", how="left", validate="many_to_one")
)
venue_cuisine_region["region_match"] = venue_cuisine_region["region_mapping_cuisine"].notna()

unmatched_region_cuisines = (
    venue_cuisine_region.loc[~venue_cuisine_region["region_match"]]
    .groupby("cuisine")["venue_id"]
    .nunique()
    .rename("venues")
    .sort_values(ascending=False)
    .to_frame()
)
print(f"Venue-cuisine pairs with a region match: {int(venue_cuisine_region['region_match'].sum()):,} / {len(venue_cuisine_region):,}")
print("Canonical cuisines without a region entry:")
display(unmatched_region_cuisines)
display(venue_cuisine_region.head(10))


Venue-cuisine pairs with a region match: 42,313 / 43,425
Canonical cuisines without a region entry:


,venues
cuisine,
grilled_food,296
barbecue,227
breakfast,129
brunch,126
fast_food,87
buffet,68
rotisserie,52
healthy,37
organic,23


,venue_id,cuisine,cuisine_key,region_mapping_cuisine,normalized_continent,normalized_region,normalized_country,normalized_cuisine,normalized_locality,osm_id,name,type,region_match
0,2,french,french,french,Europe,Western Europe,France,French,France,node/25733700,L'Esprit Bistrot,restaurant,True
1,3,regional,regional,regional,Europe,Western Europe,France,French,Regional French,node/26509939,Au Vieux Moulin,restaurant,True
2,3,french,french,french,Europe,Western Europe,France,French,France,node/26509939,Au Vieux Moulin,restaurant,True
3,5,regional,regional,regional,Europe,Western Europe,France,French,Regional French,node/26858709,Maison Las Clauzes,restaurant,True
4,7,vietnamese,vietnamese,vietnamese,Asia,Southeast Asia,Vietnam,Vietnamese,Vietnamese,node/29047143,Bistr'O bo bun,restaurant,True
5,7,japanese,japanese,japanese,Asia,East Asia,Japan,Japanese,Japanese,node/29047143,Bistr'O bo bun,restaurant,True
6,8,french,french,french,Europe,Western Europe,France,French,France,node/29681754,L'instant,restaurant,True
7,21,regional,regional,regional,Europe,Western Europe,France,French,Regional French,node/38461659,Café de la Paix,restaurant,True
8,21,french,french,french,Europe,Western Europe,France,French,France,node/38461659,Café de la Paix,restaurant,True
9,26,french,french,french,Europe,Western Europe,France,French,France,node/61948490,Le Bistro Autrement,restaurant,True


##### 1.3.3.9 Review cuisines without a regional match

List canonical cuisine labels that still need a regional mapping decision.


In [331]:
# List canonical cuisines that have no matching row in cuisine_region_mapping.csv.
cuisines_missing_region_entry = (
    venue_cuisine_region.loc[~venue_cuisine_region["region_match"]]
    .groupby("cuisine")["venue_id"]
    .nunique()
    .rename("venues")
    .sort_values(ascending=False)
    .to_frame()
)
print(f"Canonical cuisine labels without a region entry: {len(cuisines_missing_region_entry):,}")
print(f"Venue-cuisine pairs without a region entry: {int(cuisines_missing_region_entry['venues'].sum()):,}")
display(cuisines_missing_region_entry)


Canonical cuisine labels without a region entry: 30
Venue-cuisine pairs without a region entry: 1,112


,venues
cuisine,
grilled_food,296
barbecue,227
breakfast,129
brunch,126
fast_food,87
buffet,68
rotisserie,52
healthy,37
organic,23


##### 1.3.3.10 Deduplicate venue-country-cuisine rows

Build a derived table while retaining the detailed venue-cuisine region table.


In [332]:
# Deduplicate to one row per venue, normalized country, and normalized cuisine.
# Keep source labels and locality labels as joined detail, and leave venue_cuisine_region unchanged.
mapped_region_pairs = venue_cuisine_region.loc[
    venue_cuisine_region["region_match"]
    & venue_cuisine_region["normalized_country"].str.strip().ne("")
    & venue_cuisine_region["normalized_cuisine"].str.strip().ne("")
].copy()
venue_cuisine_region_grouped = (
    mapped_region_pairs
    .groupby(
        ["venue_id", "normalized_country", "normalized_cuisine"],
        as_index=False,
        dropna=False,
    )
    .agg(
        osm_id=("osm_id", "first"),
        name=("name", "first"),
        type=("type", "first"),
        normalized_continent=("normalized_continent", "first"),
        normalized_region=("normalized_region", "first"),
        source_cuisines=("cuisine", join_unique_canonical),
        normalized_localities=("normalized_locality", join_unique_canonical),
    )
)
region_grouped_output_path = DATA_DIR / "venue_cuisine_region_grouped.csv"
venue_cuisine_region_grouped.to_csv(region_grouped_output_path, index=False, encoding="utf-8-sig")
print(f"Deduplicated venue-country-cuisine rows: {len(venue_cuisine_region_grouped):,}")
print(f"Exported grouped table to {region_grouped_output_path}")
display(venue_cuisine_region_grouped.head(100))


Deduplicated venue-country-cuisine rows: 41,246
Exported grouped table to /home/m0/Documents/captain-notebook/france_geofabrik_raw_data/venue_cuisine_region_grouped.csv


,venue_id,normalized_country,normalized_cuisine,osm_id,name,type,normalized_continent,normalized_region,source_cuisines,normalized_localities
0,2,France,French,node/25733700,L'Esprit Bistrot,restaurant,Europe,Western Europe,french,France
1,3,France,French,node/26509939,Au Vieux Moulin,restaurant,Europe,Western Europe,regional;french,Regional French;France
2,5,France,French,node/26858709,Maison Las Clauzes,restaurant,Europe,Western Europe,regional,Regional French
3,7,Japan,Japanese,node/29047143,Bistr'O bo bun,restaurant,Asia,East Asia,japanese,Japanese
4,7,Vietnam,Vietnamese,node/29047143,Bistr'O bo bun,restaurant,Asia,Southeast Asia,vietnamese,Vietnamese
...,...,...,...,...,...,...,...,...,...,...
95,333,Lebanon,Lebanese,node/257109620,Adonys,restaurant,Asia,Middle East,lebanese,Lebanon
96,335,France,French,node/257109622,Le Bouchon des Carnivores,restaurant,Europe,Western Europe,regional,Regional French
97,338,Mexico,Mexican,node/257109628,Nachos,fast_food,North America,Central America,mexican,Mexican
98,339,France,French,node/257109629,Chez M'man,restaurant,Europe,Western Europe,french;regional,France;Regional French


#### 1.3.4 Drop the raw OSM tags


In [333]:
# Drop raw OSM tags when present; this cell can be rerun safely.
pbf_df = pbf_df.drop(columns=["tags"], errors="ignore")
print(f"Raw tags column retained: {'tags' in pbf_df.columns}")
display(pbf_df.columns.tolist())

Raw tags column retained: False


['X',
 'Y',
 'osm_id',
 'type',
 'name',
 'brand',
 'operator',
 'wheelchair',
 'opening_hours',
 'level',
 'siret',
 'profession_ref',
 'wikidata',
 'website',
 'phone',
 'email',
 'facebook',
 'address',
 'com_insee',
 'com_nom',
 'last_update',
 'cuisine_raw',
 'cuisine_standard',
 'dish']

#### 1.3.5 Save the cleaned Geofabrik dataset


In [334]:
# Save the cleaned Geofabrik venue dataset after dropping raw OSM tags.
clean_output_path = DATA_DIR / "food_venues_cleaned.csv"
pbf_df.to_csv(clean_output_path, sep=";", index=False, encoding="utf-8-sig")
print(f"Saved {len(pbf_df):,} cleaned Geofabrik venues to {clean_output_path}")

Saved 190,259 cleaned Geofabrik venues to /home/m0/Documents/captain-notebook/france_geofabrik_raw_data/food_venues_cleaned.csv
